# 🌏 7장 · 끼리끼리 모으기: 군집화 (개정판 실습 노트북 v12: 책 v15)

『빅데이터분석 with 파이썬』 7장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 7-1, 7-2, 7-3, 7-4, 7-5, 7-6 |
| ☆ 선택 · **코랩 실습** | 자습이나 심화. 책에는 할 일·단계·확인할 결과만 있고, 코드와 풀이는 이 노트북에만 있습니다 | 7-7, 7-8, 7-9, 7-10, 7-11 |

**데이터**
- 전 세계 규모 5 이상 지진(미국지질조사국 USGS, 1900년~): 실습 7-1이 캐글(Kaggle)에서 자동으로 받습니다(약 21MB). 안 되면 책 깃허브 저장소의 같은 파일을 받습니다.
- 미니 프로젝트: 서울교통공사 역별 시간대별 승하차 인원(2024년)을 달마다 평균 낸 파일 `서울지하철_역별_시간대별_월평균_승하차_2024.csv`. 책과 함께 나누어 주는 파일을 코랩 왼쪽 파일 창에 올려 두십시오.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다. 실습 7-6은 1~3분쯤 걸립니다.

> 🗺️ **지도로 보기** 셀은 책에는 없는 노트북 전용 덤입니다. 책 실습이 그린 세계지도 위에서 분석을 한 걸음 더 나아갑니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib kagglehub geopandas

## 작은 예제로 먼저: [예제 7-1~7-4] Ⓑ

01·04절의 [예제]를 손으로 푼 뒤 같은 계산을 코드로 따라 합니다. 실제 데이터 실습(7-1~)보다 먼저 해 보세요.

#### [예제 7-1] Ⓑ 파이썬 코드로: 지진 셋의 거리, 표준화 전후 (1.3)

책의 손 풀이와 같은 숫자가 나오는지 확인합니다. 단계(①②③)마다 한 셀입니다.

In [ ]:
# ❶
import numpy as np, pandas as pd
q = np.array([[30, 5.0], [40, 8.0], [80, 5.0]])    # 지진 1~3 (깊이, 규모)
mu = np.array([62.23, 5.446])                       # 열별 평균
sd = np.array([108.35, 0.4823])                     # 열별 표준편차

In [ ]:
# ❷
def dist(p, r):
    return np.sqrt(((p - r) ** 2).sum( ))            # (식 7-1)

z = (q - mu) / sd                                   # (식 7-2): 세 줄을 한 번에
print(z.round(3))
print("그대로 : 1-2", dist(q[0], q[1]).round(2), " 1-3", dist(q[0], q[2]).round(2))
print("표준화 : 1-2", dist(z[0], z[1]).round(2), " 1-3", dist(z[0], z[2]).round(2))

In [ ]:
# ❸
from scipy.spatial.distance import cdist
print(pd.DataFrame(cdist(z, z), index=["지진1", "지진2", "지진3"],
                   columns=["지진1", "지진2", "지진3"]).round(2))

#### [예제 7-2] Ⓑ 파이썬 코드로: 새 지진의 z와 정규분포 몫 (4.1)

책의 손 풀이와 같은 숫자가 나오는지 확인합니다. 단계(①②③)마다 한 셀입니다.

In [ ]:
# ❶
from scipy.stats import norm
new = np.array([15, 6.84])                          # 새 지진 (깊이, 규모)
z_new = (new - mu) / sd                             # (식 7-2)
print("z =", z_new.round(2))

In [ ]:
# ❷
print("깊이 15km 이하일 몫 :", round(norm.cdf(z_new[0]) * 100, 1), "%")
print("규모 6.84 이상일 몫:", round(norm.sf(z_new[1]) * 100, 2), "%")

#### [예제 7-3] Ⓑ 파이썬 코드로: k-평균 한 바퀴 표 (4.3)

책의 손 풀이와 같은 숫자가 나오는지 확인합니다. 단계(①②③)마다 한 셀입니다.

In [ ]:
# ❶
pts = np.array([[1, 1], [1.5, 2], [3, 4], [5, 7], [3.5, 5], [4.5, 5], [3.5, 4.5]])
C = pts[[0, 3]]                                     # 처음 중심: 점 1과 점 4
d2 = ((pts[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)   # 거리 제곱 표 7×2

In [ ]:
# ❷
tab = pd.DataFrame(d2, columns=["c1까지", "c2까지"], index=range(1, 8))
tab["배정"] = d2.argmin(axis=1) + 1                 # (식 7-5): 더 가까운 중심 번호
print(tab)
print("WCSS:", d2.min(axis=1).sum( ))                # 줄마다 작은 값의 합 (식 7-4)

In [ ]:
# ❸
a = d2.argmin(axis=1)
newC = np.array([pts[a == k].mean(axis=0) for k in range(2)])   # (식 7-6)
print("새 중심:", newC.round(3).tolist( ))

#### [예제 7-4] Ⓑ 파이썬 코드로: 점 3의 실루엣 (4.4)

책의 손 풀이와 같은 숫자가 나오는지 확인합니다. 단계(①②③)마다 한 셀입니다.

In [ ]:
# ❶
from sklearn.metrics import silhouette_samples
pts = np.array([[1, 1], [1.5, 2], [3, 4], [5, 7], [3.5, 5], [4.5, 5], [3.5, 4.5]])
lab = np.array([0, 0, 1, 1, 1, 1, 1])               # 멈춘 뒤의 배정
p = pts[2]                                          # 점 3 (3, 4)
same = (lab == lab[2]) & (np.arange(7) != 2)        # 같은 군집, 자기 자신 빼고
a = np.linalg.norm(pts[same] - p, axis=1).mean( )    # 우리 편과의 평균 거리
b = np.linalg.norm(pts[lab != lab[2]] - p, axis=1).mean( )   # 남의 편과 평균 거리
print("a =", round(a, 3), " b =", round(b, 3), " s =", round((b - a) / max(a, b), 3))

In [ ]:
# ❷
s = silhouette_samples(pts, lab)                     # 점마다 s
print("점별 s:", s.round(2))
print("실루엣 계수(평균):", s.mean( ).round(3))

In [ ]:
# ❸
order = np.argsort(s)
colors = np.where(lab[order] == 0, "tab:blue", "tab:orange")   # 군집별 색
plt.barh(np.arange(1, 8), s[order], color=colors)
plt.yticks(np.arange(1, 8), [f"점 {i + 1}" for i in order])
plt.axvline(s.mean( ), ls="--", c="gray")           # 평균 = 실루엣 계수
plt.xlabel("실루엣 s"); plt.show( )

그래프 그리기([예제 7-2] Ⓑ의 정규분포 그림)는 위 셀의 `z_new`로 `norm.pdf`를 그리면 됩니다.

## 05. 파이썬 실습: 지진 10만 건 묶어 보기

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 지진 데이터는 캐글(Kaggle)에서 `kagglehub`로 바로 받고, 받지 못하면 책 깃허브 저장소에 둔 같은 파일을 받습니다. 캐글 파일에는 새 지진이 더해질 수 있어 숫자가 책과 조금 다를 수 있습니다. 책과 똑같은 숫자를 보려면 실습 7-1의 `USE_KAGGLE`을 `False`로 바꾸십시오. 지도는 지오판다스(geopandas)로 그립니다. `kagglehub`와 지오판다스는 코랩에 이미 들어 있습니다. 판다스, 넘파이와 함께 기계학습 도구인 사이킷런을 씁니다. 사이킷런의 도구는 모두 `fit( )`으로 데이터를 배우고, 군집 도구는 배운 결과를 `labels_`(군집 번호)에 담아 둡니다. k-평균은 시작점을 무작위로 고르므로 `random_state=42`로 난수의 씨앗을 고정합니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [실습 예제 7-1 ★] 지진 데이터 불러오기와 정리

지진 데이터를 읽어 같은 지진의 중복 기록, 지진이 아닌 기록, 깊이나 규모가 빈 기록을 지우시오. 연대별 기록 수를 그리고, 진앙을 세계지도 위에 찍어 지진이 얼마나 몰려 있는지 세시오.

① 한글 글꼴을 설정하고 캐글에서 파일을 받아 읽습니다. ② 중복과 지진이 아닌 기록을 지웁니다. ③ 1972년과 1973년의 기록을 견주고 연대별 막대그래프를 그립니다. ④ 세계지도를 깔고 진앙을 찍은 뒤, 지구를 위도·경도 5도 칸으로 나누어 칸마다 지진 수를 셉니다.

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

USE_KAGGLE = True        # 캐글 파일에는 새 지진이 더해질 수 있다. 책과 똑같은 숫자를 보려면 False로
DATA = "Significant_Earthquakes.csv"
if USE_KAGGLE and not os.path.exists(DATA):        # ① 캐글에서 받기(약 21MB)
    try:
        import kagglehub
        path = kagglehub.dataset_download("usamabuttar/significant-earthquakes")
        DATA = os.path.join(path, "Significant_Earthquakes.csv")
    except Exception as e:
        print("캐글에서 받지 못했습니다:", type(e).__name__)
if not os.path.exists(DATA):                       # ② 안 되면 책 깃허브 저장소의 같은 파일
    urllib.request.urlretrieve("https://raw.githubusercontent.com/yeonu69/bigdata-python-book/"
                               "main/ch07/data/Significant_Earthquakes.csv", DATA)

df = pd.read_csv(DATA, index_col=0)                # 첫 열은 행 번호
print("처음 읽은 행:", len(df), " 같은 id가 다시 나온 행:", df["id"].duplicated().sum())
df["time"] = pd.to_datetime(df["time"], format="ISO8601")
# 같은 지진이 여러 번 실렸으면 가장 나중에 고친(updated) 기록 하나만 남기고, 다시 시간 순서로
df = df.sort_values("updated").drop_duplicates("id", keep="last").sort_values("time")
print("중복을 지운 뒤:", len(df))
print(df["type"].value_counts().head(4).to_string())

eq = df[(df["type"] == "earthquake")
        & df["latitude"].between(-85, 85) & df["longitude"].between(-180, 180)]
print("지진만, 위치가 맞는 것:", len(eq))
eq = eq.dropna(subset=["depth", "mag"]).copy()      # 깊이·규모가 빈 행 제외
eq["region"] = eq["place"].fillna("unknown").str.split(", ").str[-1]
print("분석할 지진:", len(eq), "건,", eq["time"].min().date(), "~", eq["time"].max().date())
print("규모를 잰 방법:", eq["magType"].nunique(), "가지,",
      eq["magType"].value_counts().head(2).to_dict())

year = eq["time"].dt.year
print("1972년:", (year == 1972).sum(), "건  1973년:", (year == 1973).sum(), "건")
small = eq["mag"] < 5.5                            # 규모 5.5 미만 = 작은 지진
print("작은 지진의 비율(%)  1972년까지:", round(small[year < 1973].mean() * 100, 1),
      " 1973년부터:", round(small[year >= 1973].mean() * 100, 1))

decade = year // 10 * 10                           # 1973 → 1970
eq.groupby(decade)["mag"].size().plot.bar(color="#9a9a9a", figsize=(6, 3))
plt.xlabel("연대")
plt.ylabel("기록된 지진 수")
plt.xticks(rotation=45)
plt.show()

import geopandas as gpd                            # 지도 데이터를 다루는 판다스
land = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
                     "v5.1.2/geojson/ne_110m_land.geojson")      # 육지 경계(Natural Earth)

def world_map(title="", ax=None):                  # 회색 대륙을 깔아 둔 지도
    if ax is None:
        ax = plt.subplots(figsize=(8, 4.2))[1]
    land.plot(ax=ax, color="#ececec", edgecolor="#9a9a9a", linewidth=0.5)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xlabel("경도")
    ax.set_ylabel("위도")
    ax.set_title(title)
    return ax

ax = world_map("규모 5 이상 지진의 진앙")
ax.scatter(eq["longitude"], eq["latitude"], s=0.5, color="#E04B3C", alpha=0.3)
plt.show()

cell = (eq["latitude"] // 5).astype(int).astype(str) + "," + (eq["longitude"] // 5).astype(int).astype(str)
cnt = cell.value_counts()                          # 위도·경도 5도 칸(36 × 72 = 2,592칸)마다 지진 수
half = (cnt.cumsum() < len(eq) / 2).sum() + 1      # 많은 칸부터 더해 절반을 채우는 칸 수
print("지진이 난 칸:", len(cnt), "/ 2592  지진 절반이 몰린 칸:", half, "칸")

[표 7-1]의 정리 과정을 그대로 따라 102,144건이 남았습니다. 1972년 619건이 1973년 1,362건으로 늘었고, 작은 지진의 비율은 14.3%에서 71.9%로 뛰었습니다. 작은 지진이 그때부터 생긴 것이 아니라, 그때부터 기록되기 시작한 것입니다. `region`은 "101 km ENE of Masohi, Indonesia"처럼 적힌 장소에서 마지막 쉼표 뒤, 곧 나라나 지역 이름만 떼어 낸 것입니다.

[그림 @7-1:2]의 지진은 지구 전체에 고르게 흩어지지 않고 가는 띠를 이룹니다. 지구 겉껍질을 이루는 판과 판이 만나는 경계입니다. 숫자로도 그렇습니다. 5도 칸 2,592개 가운데 지진이 한 번이라도 난 칸은 1,089개이고, 지진의 절반이 겨우 42칸(1.6%)에 몰려 있습니다. "가까운 것끼리 무리를 이룬다"는 군집화의 전제가 지도에서 눈에 보이는 셈입니다. 5도 칸은 극지방으로 갈수록 좁아지므로 대략의 숫자로 읽으십시오.

> **[코드 읽기] `drop_duplicates("id", keep="last")`**
>
> 같은 id가 여러 줄이면 마지막 줄 하나만 남기라는 뜻입니다. 바로 앞에서 고친 시각(updated) 순으로 줄을 세워 두었으므로, 마지막 줄이 가장 최근에 고친 기록입니다. 중복을 지운 뒤에는 다시 일어난 시각(time) 순으로 세웁니다.

> **[코드 읽기] `kagglehub.dataset_download( )`와 `gpd.read_file( )`**
>
> `kagglehub.dataset_download("만든이/데이터셋")`은 캐글 데이터셋을 내려받고 파일이 놓인 폴더 경로를 돌려줍니다. 받지 못하면 오류가 나므로 `try`로 감싸 두고, 그때는 저장소의 파일을 받게 했습니다. `gpd.read_file( )`은 지도 파일을 읽어 행마다 모양(geometry)이 든 표를 만듭니다. 여기서는 공개 지도 자료인 Natural Earth의 육지 경계(약 0.1MB)를 읽었습니다. 표의 `plot( )`이 육지를 회색으로 칠하고, 그 위에 `scatter(경도, 위도)`로 점을 찍습니다. 함수 `world_map( )`은 뒤의 실습에서도 씁니다.

#### [실습 예제 7-2 ★] 깊이와 규모 살펴보기

깊이와 규모의 기초 통계를 구하고, 깊이를 세 구간으로 나눈 비율, 10km와 33km로 적힌 비율, 상관계수를 구하시오. 두 분포도 그리시오.

In [ ]:
print(eq[["depth", "mag"]].describe().round(2))
depth_class = pd.cut(eq["depth"], [-10, 70, 300, 800],
                     labels=["얕은(70km 이하)", "중간(70~300km)", "깊은(300km 초과)"])
print((depth_class.value_counts(normalize=True) * 100).round(1).to_string())
d = eq["depth"].round(0)
print("깊이가 정확히 10km:", round((d == 10).mean() * 100, 1), "%,  33km:",
      round((d == 33).mean() * 100, 1), "%")
print("깊이와 규모의 상관계수:", round(eq["depth"].corr(eq["mag"]), 3))
late = eq[year >= 1973]                            # 기록이 고르게 된 뒤
print("1973년부터  규모 6 이상:", (late["mag"] >= 6).sum(), "건, 7 이상:", (late["mag"] >= 7).sum(), "건")
print(pd.cut(eq["depth"], range(200, 701, 50)).value_counts().sort_index().to_string())

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].hist(eq["depth"], bins=range(0, 720, 20), color="#9a9a9a")
ax[0].set_yscale("log")                            # 깊은 쪽의 작은 봉우리가 보이도록
ax[0].set_xlabel("깊이(km)")
ax[0].set_ylabel("지진 수(로그 눈금)")
ax[1].hist(eq["mag"], bins=np.arange(5, 9.6, 0.1), color="#E04B3C")
ax[1].set_xlabel("규모")
plt.tight_layout()
plt.show()

깊이는 평균(62.23)이 가운데 값(33.00)의 두 배 가까이 되는, 오른쪽으로 꼬리가 긴 분포입니다. 표준편차는 깊이가 108.35, 규모가 0.48로 200배 넘게 차이 납니다. 깊이 구간 표에서는 250~300km(435건)와 300~350km(253건)가 골짜기이고, 550~600km(1,294건)에 봉우리가 하나 더 있습니다. 깊이의 최솟값 −3km는 해수면보다 높은 산 아래의 지진을 해수면 기준으로 잰 것입니다.

> **[코드 읽기] `pd.cut(값, 경계)`**
>
> 숫자를 구간으로 나눕니다. 경계가 [−10, 70, 300, 800]이면 "−10 초과 70 이하", "70 초과 300 이하", "300 초과 800 이하"의 세 구간입니다. `value_counts(normalize=True)`는 개수 대신 비율을 냅니다.

### 🗺️ [지도로 보기 1] 육지인가 바다인가, 깊이별로는

실습 7-1에서 만든 `land`(육지 다각형)와 `world_map( )`을 다시 씁니다. 진앙(점)이 육지 다각형 안에 있는지 가려 육지·바다 비율을 구하고, 실습 7-2의 세 깊이 구간으로 색을 칠해 지도에 찍습니다. 얕은 지진을 먼저, 깊은 지진을 나중에 그려 수가 적은 깊은 지진이 묻히지 않게 합니다.

In [ ]:
# ❶ 진앙이 육지 위인가: 점이 육지 다각형 안에 있으면 True
geo = gpd.GeoDataFrame(geometry=gpd.points_from_xy(eq["longitude"], eq["latitude"]), crs=land.crs)
inside = gpd.sjoin(geo, land[["geometry"]], predicate="within").index.unique()
eq["육지"] = np.isin(np.arange(len(eq)), inside)
print("육지 위의 진앙:", round(eq["육지"].mean() * 100, 1), "%,  바다 위:", round((~eq["육지"]).mean() * 100, 1), "%")

# ❷ 깊이 구간별 색 지도
DCOL = {"얕은(70km 이하)": "#E04B3C", "중간(70~300km)": "#2E9E5B", "깊은(300km 초과)": "#5B3FA0"}
ax = world_map("깊이별 지진 위치")
for c in DCOL:                                     # 얕은 → 중간 → 깊은 순서로 그려 깊은 것이 위에
    part = eq[depth_class == c]
    ax.scatter(part["longitude"], part["latitude"], s=1 if c.startswith("얕은") else 3,
               color=DCOL[c], alpha=0.5, label=f"{c} {len(part):,}건")
ax.legend(markerscale=6, loc="lower left")
plt.show()

# ❸ 깊이 구간별 육지 비율과 많은 지역
print("깊이 구간별 육지 위 진앙의 비율(%):",
      (eq.groupby(depth_class, observed=True)["육지"].mean() * 100).round(1).to_dict())
for c in ["중간(70~300km)", "깊은(300km 초과)"]:
    print(c, "많은 지역:", eq.loc[depth_class == c, "region"].value_counts().head(4).to_dict())

🔍 **지도 읽기**: 진앙의 77.9%가 바다 위입니다. 얕은 지진(빨강)은 모든 띠에 있지만, 깊은 지진(보라)은 태평양 서쪽의 피지·통가, 인도네시아, 일본 아래, 그리고 남아메리카 안쪽에 주로 모여 있습니다. 바다의 판이 다른 판 밑으로 비스듬히 파고드는 곳(섭입대)입니다. 파고든 판을 따라 지진이 일어나므로 해구에서 육지 쪽으로 갈수록 깊어집니다. 남아메리카에서 해안의 빨강, 안쪽의 초록, 더 안쪽의 보라가 차례로 놓인 것이 그 흔적입니다.

깊은 지진 가운데 육지 위 진앙은 6.3%뿐이고, 많은 지역 1~3위가 모두 피지 둘레(합쳐서 2,364건)입니다. 중간 깊이는 육지 위가 34.0%로 가장 높습니다. 파고든 판이 육지 밑까지 들어간 뒤 70~300km 깊이에서 지진을 일으키기 때문입니다. 지도가 1:1억 1천만 축척이라 해안선이 거칠어, 해안 가까운 진앙은 육지·바다가 바뀌어 셀 수 있습니다.

#### [실습 예제 7-3 ★] k-평균을 손으로 돌려 보기

[예제 7-2]의 점 일곱 개를 넘파이로 두 군집으로 나누되, 배정과 갱신을 직접 코드로 쓰시오.

`pts[:, None, :] - C[None, :, :]`는 점 7개와 중심 2개의 모든 짝에 대해 좌표 차이를 한 번에 구합니다. 2장의 브로드캐스팅입니다. 결과는 7 × 2 × 2 모양이고, 마지막 축을 따라 제곱해 더하면 7 × 2의 거리 제곱 표가 됩니다.

In [ ]:
pts = np.array([[1, 1], [1.5, 2], [3, 4], [5, 7], [3.5, 5], [4.5, 5], [3.5, 4.5]])
C = pts[[0, 3]].copy()                             # 처음 중심: 점 1과 점 4
for step in range(1, 10):
    d2 = ((pts[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)   # 점-중심 거리의 제곱
    a = d2.argmin(axis=1)                          # ① 배정: 가장 가까운 중심
    print(f"{step}회: 중심 {C.round(2).tolist()}, 배정 {a.tolist()}, "
          f"WCSS {d2.min(axis=1).sum():.2f}")
    newC = np.array([pts[a == j].mean(axis=0) for j in range(2)])  # ② 갱신: 평균
    if np.allclose(newC, C):                       # 중심이 그대로면 멈춤
        break
    C = newC

plt.figure(figsize=(4, 4))
plt.scatter(pts[:, 0], pts[:, 1], c=np.where(a == 1, "#E04B3C", "#9a9a9a"), s=80)
plt.scatter(C[:, 0], C[:, 1], marker="x", s=150, color="black")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

손 계산과 같습니다. 2회째에 세 번째 점 (3, 4)의 배정이 0에서 1로 바뀌고, 3회째에는 중심이 그대로라 멈춥니다. `argmin( )`은 거리가 같으면 앞의 중심을 고르므로, 1회째에 (3, 4)가 첫 번째 군집에 들어갔습니다.

#### [실습 예제 7-4 ★] 거리와 표준화

1.3의 지진 세 개로 표준화 전 거리를 구하고, 실제 데이터로 표준화 도구를 만든 뒤 표준화 후 거리를 구하시오. 표준화 없이 k = 3으로 묶은 중심도 구하시오.

In [ ]:
def dist(p, q):                                    # 유클리드 거리
    return np.sqrt(((p - q) ** 2).sum())

q = np.array([[30, 5.0], [40, 8.0], [80, 5.0]])    # [깊이(km), 규모] 지진 1, 2, 3
print("표준화 전  1-2:", round(dist(q[0], q[1]), 2), " 1-3:", round(dist(q[0], q[2]), 2))

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = eq[["depth", "mag"]].to_numpy()
scaler = StandardScaler().fit(X)                   # 평균과 표준편차를 배운다
print("평균:", scaler.mean_.round(2), " 표준편차:", scaler.scale_.round(2))
z = scaler.transform(q)                            # (식 7-3)을 적용한다
print("표준화 후  1-2:", round(dist(z[0], z[1]), 2), " 1-3:", round(dist(z[0], z[2]), 2))
print("깊이 15km, 규모 6.84인 지진의 z:", scaler.transform([[15, 6.84]]).round(2))

raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)   # 표준화 없이 묶으면
for depth, mag in sorted(raw.cluster_centers_.tolist()):
    print(f"표준화 없이 묶은 중심: 깊이 {depth:6.1f}km, 규모 {mag:.2f}")

plt.figure(figsize=(6, 3))
plt.scatter(X[::20, 0], X[::20, 1], c=raw.labels_[::20], cmap="Greys", vmin=-1, s=6)
plt.xlabel("깊이(km)")
plt.ylabel("규모")
plt.show()

표준화 전에는 지진 2가, 뒤에는 지진 3이 지진 1과 가깝습니다. 표준화 없이 묶은 세 중심은 규모가 5.37~5.47로 거의 같습니다. 깊이만 보고 나눈 것입니다. 이 결과로는 규모 9의 대지진과 규모 5의 지진이 같은 군집에 섞입니다.

> **[코드 읽기] `fit( )`, `transform( )`, `inverse_transform( )`**
>
> `StandardScaler( ).fit(X)`는 열마다 평균과 표준편차를 배워 `mean_`과 `scale_`에 담습니다. `transform( )`은 배운 값으로 (식 7-3)을 적용하고, `inverse_transform( )`은 거꾸로 원래 단위로 되돌립니다. 배우기와 적용하기가 나뉘어 있어서, 새 지진이 와도 같은 눈금으로 바꿀 수 있습니다.

#### [실습 예제 7-5 ★] k-평균으로 세 유형 찾기

표준화한 깊이와 규모로 k = 3인 k-평균을 돌리고, 군집마다 지진 수, 깊이와 규모의 모습을 구하시오. 시대별 구성비와 정해 넣은 깊이의 비율도 구하시오. 세 유형을 세계지도 위에도 찍으시오.

`n_init=10`은 시작점을 바꿔 10번 돌린 뒤 군집 내 제곱합이 가장 작은 결과를 고르라는 뜻입니다. 군집 번호에는 뜻이 없으므로, 중심을 원래 단위로 되돌려 보고 사람이 이름을 붙이는 함수 `name( )`을 만들었습니다.

In [ ]:
Xs = scaler.transform(X)
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Xs)
centers = scaler.inverse_transform(km.cluster_centers_)   # 원래 단위로 되돌리기
print("표준화된 중심:", km.cluster_centers_.round(2).tolist())
print("원래 단위의 중심:", centers.round(2).tolist())

def name(c):                                       # 중심을 보고 사람이 붙이는 이름
    return "C 깊은" if c[0] > 200 else ("B 얕고 강한" if c[1] > 5.7 else "A 얕고 보통")

eq["군집"] = [name(centers[l]) for l in km.labels_]
summary = eq.groupby("군집").agg(지진수=("mag", "size"), 깊이평균=("depth", "mean"),
                                 깊이최소=("depth", "min"), 규모평균=("mag", "mean"),
                                 규모최소=("mag", "min"), 규모최대=("mag", "max"))
summary["비율%"] = (summary["지진수"] / len(eq) * 100).round(1)
print(summary.round(2).to_string())

period = np.where(year < 1973, "1972년까지", "1973년부터")   # 기록이 크게 는 해
print((eq.groupby(period)["군집"].value_counts(normalize=True) * 100).round(1).unstack())
fixed = d.isin([10, 33])                           # 관례로 정해 넣은 깊이
print("정해 넣은 깊이의 비율(%):", (fixed.groupby(eq["군집"]).mean() * 100).round(1).to_dict())

COLOR = {"A 얕고 보통": "#9a9a9a", "B 얕고 강한": "#E04B3C", "C 깊은": "#222222"}
s = eq.sample(3000, random_state=1)
plt.figure(figsize=(6, 3.5))
for g, part in s.groupby("군집"):
    plt.scatter(part["depth"], part["mag"], s=6, color=COLOR[g], label=g)
plt.xlabel("깊이(km)")
plt.ylabel("규모")
plt.legend(markerscale=3)
plt.show()

[표 7-2]의 숫자가 그대로 나옵니다. C의 가장 얕은 지진은 276.2km입니다. 표준화된 중심 (−0.24, 1.42)만 보고는 뜻을 알기 어렵지만, 깊이 35.7km, 규모 6.13으로 되돌리면 "얕고 강한 지진"이라는 이름이 떠오릅니다. 뒤의 두 표는 [표 7-5]의 숫자입니다. 1972년까지의 기록에서는 B가 69.1%로 가장 많고, 1973년부터는 A가 79.2%입니다.

**❺ 지도로 보기: 세 유형을 세계지도에 찍기**

군집화에는 깊이와 규모만 넣었고 위치(경도·위도)는 넣지 않았습니다. 그 결과를 위치 위에 찍어 봅니다. `world_map( )`은 실습 예제 7-1에서 만든 지도 함수입니다. A → B → C 순서로 찍어 수가 가장 적은 C가 맨 위에 보이게 합니다.

In [ ]:
ax = world_map("세 유형의 진앙")                   # 실습 예제 7-1의 지도 함수
for g in COLOR:                                    # A → B → C 순서로 찍어 C가 맨 위에
    part = eq[eq["군집"] == g]
    ax.scatter(part["longitude"], part["latitude"], s=1, color=COLOR[g], alpha=0.5, label=g)
ax.legend(markerscale=6, loc="lower left")
plt.show()
lon = eq.loc[eq["군집"] == "C 깊은", "longitude"]
print("C 가운데 태평양 서쪽:", round(((lon >= 100) | (lon <= -170)).mean() * 100, 1), "%")

위 지도에서 A와 B는 같은 띠 위에 있습니다. 같은 곳에서 일어나는 지진을 규모로 나눈 것입니다. C는 전혀 다른 곳에 모입니다. 95.2%가 동경 100도에서 서경 170도 사이, 곧 태평양 서쪽의 피지·통가, 인도네시아, 일본 아래에 있고, 나머지는 남아메리카 안쪽에 있습니다. 바다의 판이 다른 판 밑으로 깊이 파고드는 곳입니다. k-평균에 위치를 넣지 않았는데도 C가 지도에서 따로 모였으니, "깊은 지진"이라는 이름이 지구의 실제 구조와 맞닿아 있다는 증거입니다. 군집에 쓰지 않은 정보로 이름을 확인하는 이 방법은 미니 프로젝트에서도 씁니다.

#### [실습 예제 7-6 ★] 몇 개로 나눌까, 진짜일까

k를 2부터 8까지 바꿔 가며 군집 내 제곱합과 실루엣 계수를 구하시오. 무리가 없는 두 데이터와도 실루엣을 견주시오.

실루엣은 계산량이 많아 1만 건을 뽑아 계산합니다. `KMeans`는 군집 내 제곱합을 `inertia_`에 담아 둡니다. `rng.uniform( )`은 같은 범위에 점을 고르게 뿌리고, `rng.permutation( )`은 한 열의 순서를 뒤섞습니다. 이 셀은 2~3분 걸릴 수 있습니다.

In [ ]:
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(42)
samp = rng.choice(len(Xs), 10000, replace=False)    # 실루엣은 1만 건 표본으로
rows = []
for k in range(2, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    rows.append([k, m.inertia_, silhouette_score(Xs[samp], m.labels_[samp])])
    if k == 2:
        print("k=2의 두 군집:", sorted(np.bincount(m.labels_).tolist(), reverse=True))
score = pd.DataFrame(rows, columns=["k", "WCSS", "실루엣"])
print(score.round({"WCSS": 0, "실루엣": 3}).to_string(index=False))

# 무리가 없는 데이터와 견주기 (k = 3)
U = rng.uniform(Xs.min(axis=0), Xs.max(axis=0), size=Xs.shape)   # 같은 범위에 고르게 뿌린 점
S = np.column_stack([rng.permutation(Xs[:, 0]),                 # 깊이와 규모를 따로 섞어
                     rng.permutation(Xs[:, 1])])                # 둘의 짝을 끊은 지진
for label, data in [("실제 지진", Xs), ("고르게 뿌린 점", U), ("따로 섞은 지진", S)]:
    m = KMeans(n_clusters=3, n_init=10, random_state=42).fit(data)
    sizes = np.sort(np.bincount(m.labels_)) / len(data) * 100
    print(f"{label:8s} 실루엣 {silhouette_score(data[samp], m.labels_[samp]):.3f}"
          f"  군집 크기(%) {sizes.round(1).tolist()}")

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].plot(score["k"], score["WCSS"], "o-", color="#555555")
ax[0].set_xlabel("k")
ax[0].set_ylabel("WCSS")
ax[1].plot(score["k"], score["실루엣"], "o-", color="#E04B3C")
ax[1].set_xlabel("k")
ax[1].set_ylabel("실루엣 계수")
plt.tight_layout()
plt.show()

[표 7-3]과 [표 7-4]의 숫자입니다. 고르게 뿌린 점도 27.9%, 27.9%, 44.2%로 세 무리가 되었습니다(1.1). 실루엣은 실제 지진 0.585, 고르게 뿌린 점 0.390이고, 따로 섞은 지진은 0.587로 실제와 같습니다. 따로 섞은 지진의 군집 크기도 4.3%, 23.9%, 71.8%로 실제와 같습니다. 세 유형이 각 특징의 분포 모양에서 나왔다는 3.6의 해석을 뒷받침합니다.

#### [실습 예제 7-7 ☆] 결정나무로 군집 설명하기 (코랩 실습)

군집 이름을 정답으로 놓고 깊이 2인 결정나무를 학습시켜, 군집을 나누는 규칙과 그 규칙이 맞힌 비율을 구하시오.

> 💻 **코랩 실습**: 책에는 이 실습의 할 일·단계·확인할 결과만 실려 있습니다. 코드와 풀이는 이 노트북에서 보십시오.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(eq[["depth", "mag"]], eq["군집"])
print(export_text(tree, feature_names=["depth", "mag"], decimals=1))
print("군집을 맞힌 비율:", round(tree.score(eq[["depth", "mag"]], eq["군집"]) * 100, 2), "%")

위가 결정나무가 찾은 규칙입니다. 결정나무는 표준화하지 않은 원래 단위로 학습시켰습니다. 그래서 "규모 5.7", "깊이 284km"처럼 바로 읽을 수 있는 기준이 나옵니다. 결정나무는 10장에서 자세히 배웁니다.

#### [실습 예제 7-8 ☆] 깊이에 로그를 씌우면 (코랩 실습)

깊이에 로그를 씌워 k = 3으로 다시 묶고, 원래 군집과 얼마나 겹치는지 구하시오.

> 💻 **코랩 실습**: 책에는 이 실습의 할 일·단계·확인할 결과만 실려 있습니다. 코드와 풀이는 이 노트북에서 보십시오.

In [ ]:
from sklearn.metrics import adjusted_rand_score

eq["log깊이"] = np.log1p(eq["depth"].clip(lower=0))   # log(1 + 깊이)
XL = StandardScaler().fit_transform(eq[["log깊이", "mag"]])
kl = KMeans(n_clusters=3, n_init=10, random_state=42).fit(XL)
print(eq.groupby(kl.labels_).agg(지진수=("mag", "size"), 깊이중앙값=("depth", "median"),
                                 규모평균=("mag", "mean")).round(2).to_string())
print("원래 군집과 겹치는 정도(조정 랜드 지수):", round(adjusted_rand_score(km.labels_, kl.labels_), 2))

In [ ]:
# ❸ 교차표: 원래 유형(행)의 지진이 새 군집(열)의 어디로 갔나
print(eq["depth"].skew( ).round(2), eq["log깊이"].skew( ).round(2))   # 꼬리의 길이(왜도): 로그로 줄어든다
print(pd.crosstab(eq["군집"], kl.labels_))

로그를 씌우면 얕은 지진 안의 작은 차이가 커집니다. 그래서 깊이 중앙값 24km 근처의 두 무리와 114km 근처의 무리로 나뉩니다. 두 결과가 얼마나 같은지는 **조정 랜드 지수**(adjusted rand index)로 쟀습니다. 두 군집 결과에서 "같은 무리로 묶인 짝"이 얼마나 일치하는지를 0(우연 수준)에서 1(완전히 같음) 사이로 나타낸 값입니다. 0.49이니 절반쯤만 같습니다. 어느 쪽이 맞는다기보다, 무엇을 보고 싶은지에 따라 특징을 고르는 것이 분석가의 일입니다.

#### [예제 7-5] Ⓑ 파이썬 코드로: 점 여섯 개로 와드 계층 군집 따라 하기

책의 손 풀이와 같은 숫자가 나오는지 확인합니다. 단계(①②③)마다 한 셀입니다.

In [ ]:
# ❶
import numpy as np
import matplotlib.pyplot as plt
P = np.array([[1, 1], [2, 1], [1, 2], [6, 5], [7, 5], [6, 7]])   # 점 A~F
names = list("ABCDEF")
groups = [[i] for i in range(6)]                    # 처음엔 점 하나가 군집 하나

def wcss(idx):                                      # 한 군집의 제곱합 (식 7-4)
    G = P[idx]
    return ((G - G.mean(axis=0)) ** 2).sum( )

In [ ]:
# ❷
def cost(g, h):                                     # 합치면 제곱합이 얼마나 느나
    return wcss(g + h) - wcss(g) - wcss(h)

while len(groups) > 1:
    pairs = [(i, j) for i in range(len(groups)) for j in range(i + 1, len(groups))]
    i, j = min(pairs, key=lambda ij: cost(groups[ij[0]], groups[ij[1]]))
    inc = cost(groups[i], groups[j])
    merged = groups[i] + groups[j]
    print("".join(names[k] for k in groups[i]), "+", "".join(names[k] for k in groups[j]),
          " 늘어난 제곱합", round(inc, 3))
    groups = [g for k, g in enumerate(groups) if k not in (i, j)] + [merged]

In [ ]:
# ❸
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
Z = linkage(P, method="ward")                       # 사이파이로 같은 일
print(Z[:, 2].round(3))                             # 높이 = √(2 × 늘어난 제곱합)
print(fcluster(Z, 2, criterion="maxclust"))         # 두 군집으로 자르기
dendrogram(Z, labels=names); plt.ylabel("합칠 때의 높이"); plt.show( )

#### [실습 예제 7-9 ☆] 계층 군집으로 지역 묶기 (코랩 실습)

지진이 가장 많은 12개 지역의 깊이와 규모 모습을 구하고, 와드 방법의 계층 군집으로 세 무리로 나눈 뒤 덴드로그램을 그리시오.

> 💻 **코랩 실습**: 책에는 이 실습의 할 일·단계·확인할 결과만 실려 있습니다. 코드와 풀이는 이 노트북에서 보십시오.

와드 방법(Ward's method)은 두 무리를 합쳤을 때 군집 내 제곱합이 가장 적게 늘어나는 짝부터 합칩니다. 사이파이(SciPy)의 `linkage( )`가 합치는 순서를 계산하고, `fcluster( )`가 원하는 무리 수에서 나무를 자릅니다.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

top12 = eq["region"].value_counts().head(12).index
sub = eq[eq["region"].isin(top12)]
prof = sub.groupby("region").agg(깊이평균=("depth", "mean"),
                                 중간깊이비율=("depth", lambda s: ((s > 70) & (s <= 300)).mean()),
                                 깊은비율=("depth", lambda s: (s > 300).mean()),
                                 규모평균=("mag", "mean")).loc[top12]
Z = linkage(StandardScaler().fit_transform(prof), method="ward")
prof["무리"] = fcluster(Z, t=3, criterion="maxclust")
print(prof.round(3).sort_values("무리").to_string())
plt.figure(figsize=(6, 4))
dendrogram(Z, labels=prof.index.tolist(), orientation="right", color_threshold=0,
           above_threshold_color="#555555")
plt.xlabel("합칠 때의 거리")
plt.show()

6.2에서 말한 세 무리입니다. 무리 1은 깊이 평균이 60~74km, 무리 3은 중간 깊이 지진의 비율이 8~10%입니다. 네 특징은 단위가 서로 다르므로 여기서도 먼저 표준화했습니다.

#### [예제 7-6] Ⓑ 파이썬 코드로: 점 열 개로 DBSCAN 따라 하기

책의 손 풀이와 같은 숫자가 나오는지 확인합니다. 단계(①②③)마다 한 셀입니다.

In [ ]:
# ❶
from scipy.spatial.distance import cdist
Q = np.array([[1, 1], [1.5, 1.5], [1, 2], [2, 1.2], [2.9, 1.4],
              [5, 5], [5.5, 5.2], [5.2, 6], [3.5, 3.5], [8, 1]])   # 점 1~10
eps, min_samples = 1.2, 3
D = cdist(Q, Q)                                     # 10×10 거리 표
nb = (D <= eps).sum(axis=1)                         # 반지름 안의 점 수(자기 포함)
print("이웃 수:", nb)

In [ ]:
# ❷
core = nb >= min_samples                            # 핵심점
near_core = ((D <= eps) & core[None, :]).any(axis=1)  # 반지름 안에 핵심점이 있나
kind = np.where(core, "핵심", np.where(near_core, "경계", "노이즈"))
print(pd.Series(kind, index=range(1, 11)).to_dict( ))

In [ ]:
# ❸
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=eps, min_samples=min_samples).fit(Q)
print("군집 번호:", db.labels_)                       # -1은 노이즈

#### [실습 예제 7-10 ☆] DBSCAN으로 지진 밀집 지대 찾기 (코랩 실습)

규모 6 이상인 지진의 위치로 DBSCAN을 돌려, 밀집 지대의 수와 노이즈 비율, 가장 큰 지대 다섯 곳을 구하시오. 밀집 지대를 세계지도 위에 그리시오.

> 💻 **코랩 실습**: 책에는 이 실습의 할 일·단계·확인할 결과만 실려 있습니다. 코드와 풀이는 이 노트북에서 보십시오.

지구는 둥글어서 위도와 경도의 차이로 거리를 재면 북극 쪽으로 갈수록 틀립니다. 그래서 둥근 지구 위의 거리인 하버사인 거리(haversine distance)를 씁니다. 좌표는 라디안으로, 거리 기준 `eps`는 "km ÷ 지구 반지름(6,371km)"으로 넣습니다.

In [ ]:
from sklearn.cluster import DBSCAN

big = eq[eq["mag"] >= 6].copy()                    # 규모 6 이상
rad = np.radians(big[["latitude", "longitude"]])
db = DBSCAN(eps=150 / 6371, min_samples=40,        # 반지름 150km 안에 40개 이상
            metric="haversine", algorithm="ball_tree").fit(rad)
big["zone"] = db.labels_                           # -1은 노이즈
print("규모 6 이상 지진:", len(big), "건")
print("찾은 밀집 지대:", big.loc[big.zone >= 0, "zone"].nunique(), "곳",
      " 노이즈 비율:", round((big.zone == -1).mean() * 100, 1), "%")
size = big[big.zone >= 0].groupby("zone").agg(n=("mag", "size"),
                                              지역=("region", lambda s: s.mode()[0]))
print(size.sort_values("n", ascending=False).head(5).to_string(index=False))
biggest = size["n"].idxmax()                       # 가장 큰 지대에는 어느 지역이?
print(big.loc[big.zone == biggest, "region"].value_counts().head(3).to_dict())

ax = world_map("DBSCAN 밀집 지대 (규모 6 이상)")
noise, zone = big[big.zone < 0], big[big.zone >= 0]
ax.scatter(noise["longitude"], noise["latitude"], s=3, color="#9a9a9a")     # 노이즈는 회색
ax.scatter(zone["longitude"], zone["latitude"], s=5, c=zone["zone"], cmap="tab20")
plt.show()

위가 밀집 지대의 수, 노이즈 비율, 큰 지대 다섯 곳의 숫자입니다. 코드가 그리는 지도도 책 6.3의 지도와 같은 모양인데, 노이즈는 회색, 밀집 지대는 지대마다 다른 색으로 칠합니다. 지대가 모두 판 경계의 띠를 따라 길게 늘어서 있습니다. 가장 큰 지대는 파푸아뉴기니와 바누아투, 솔로몬 제도에 걸쳐 있어, 그 안에서 가장 많이 나온 지역 이름(최빈값)으로 표시했습니다. `eps`와 `min_samples`를 바꾸면 결과가 크게 달라집니다. [연습문제 7-6]에서 확인해 봅니다.

### 🗺️ [지도로 보기 2] 밀집 지대는 얼마나 길까

실습 7-10의 지도에서 밀집 지대는 띠를 따라 길게 늘어서 있었습니다. 큰 지대 다섯 곳의 남북·동서 길이를 재고, 노이즈가 어디에 있는지 봅니다. 통가와 알래스카 지대는 경도 180도 선을 넘으므로 경도 폭을 잴 때 조심합니다.

In [ ]:
def span(lon):                                     # 180도 선을 넘는 지대도 바르게 재는 경도 폭
    return min(lon.max() - lon.min(), (lon % 360).max() - (lon % 360).min())

ext = zone.groupby("zone").apply(lambda p: pd.Series({
    "지역": p["region"].mode()[0], "지진수": len(p),
    "남북km": round((p["latitude"].max() - p["latitude"].min()) * 111),       # 위도 1도 ≈ 111km
    "동서km": round(span(p["longitude"]) * 111 * np.cos(np.radians(p["latitude"].mean())))}))
print(ext.sort_values("지진수", ascending=False).head(5).to_string(index=False))
print("노이즈 가운데 바다 위 진앙:", round((~noise["육지"]).mean() * 100, 1), "%")

🔍 **지도 읽기**: 가장 큰 파푸아뉴기니 지대는 남북 2,493km, 동서 3,528km, 일본 지대는 남북 2,789km에 걸칩니다. k-평균이었다면 둥근 덩어리로 잘랐을 띠 모양을 DBSCAN은 "이웃이 충분히 많은 점을 이어 가는" 방식이라 그대로 따라갑니다. 통가와 알래스카 지대는 지도에서 양 끝으로 갈라져 보이지만, 하버사인 거리 덕분에 하나로 묶였습니다.

노이즈 5,225건도 대부분(62.1%) 바다 위의 띠에 있습니다. 지진이 없는 곳이 아니라, 반지름 150km 안에 규모 6 이상이 40건을 넘지 못한 성긴 띠입니다. 대서양 가운데 띠가 통째로 노이즈인 것이 그 예입니다.

#### [예제 7-7] Ⓑ 파이썬 코드로: 점 네 개로 PCA 따라 하기

책의 손 풀이와 같은 숫자가 나오는지 확인합니다. 단계(①②③)마다 한 셀입니다.

In [ ]:
# ❶
X = np.array([[-2, -1], [-1, 1], [1, -1], [2, 1]])  # 평균이 (0, 0)인 네 점
C = np.cov(X.T, bias=True)                          # n으로 나눈 공분산 행렬
print(C)

In [ ]:
# ❷
lam, U = np.linalg.eigh(C)                          # 고윳값(작은 순), 고유벡터
lam, U = lam[::-1], U[:, ::-1]                       # 큰 순서로 뒤집기
print("고윳값:", lam.round(3), " 몫:", (lam / lam.sum( )).round(3))
print("1주성분 방향:", U[:, 0].round(3))
print("그림자 좌표:", (X @ U[:, 0]).round(2))

In [ ]:
# ❸
from sklearn.decomposition import PCA
pca = PCA(n_components=1).fit(X)
print("사이킷런 몫:", pca.explained_variance_ratio_.round(3),
      " 방향:", pca.components_[0].round(3))

#### [실습 예제 7-11 ☆] 40칸짜리 하루 리듬을 두 축 지도로 (코랩 실습)

미니 프로젝트의 지하철역 240곳을 승하차 40칸으로 나타내고, PCA로 두 축까지 줄여 지도 한 장에 그리시오. 두 축으로 묶은 군집이 40칸으로 묶은 군집과 같은지 견주시오. PCA의 원리는 06절의 6.4에서 자세히 봅니다.

> 💻 **코랩 실습**: 책에는 이 실습의 할 일·단계·확인할 결과만 실려 있습니다. 코드와 풀이는 이 노트북에서 보십시오.

**① 문제 정의하기: 40칸짜리 하루 리듬을 2차원 평면에 그릴 수 있을까요?**

미니 프로젝트에서는 서울 지하철역 240곳을 평일 하루 리듬으로 묶습니다. 한 역은 승차 20칸, 하차 20칸, 모두 40칸입니다. 지진은 두 특징만 써서 평면에 바로 그릴 수 있었지만, 40칸은 그릴 수 없습니다. 40칸의 정보를 최대한 살린 새 축 두 개를 만들어, 240개 역을 지도 한 장에 그려 봅시다. 그 지도에서도 세 유형이 갈라져 보인다면 PCA의 효과를 눈으로 확인할 수 있을 것입니다.

**문제 해결에 필요한 정보를 미리 살펴봅시다.**

1. 이 활동에 필요한 데이터는 무엇이고, 어디에서 구할 수 있나요? → 미니 프로젝트와 같은 "서울지하철_역별_시간대별_월평균_승하차_2024.csv"입니다. 강의 자료로 나누어 주고, 책 깃허브 저장소에도 있습니다. 평일의 시간대별 승하차 40칸을 모두 씁니다.

2. 모델 학습에 쓸 알고리즘은 무엇인가요? → PCA(주성분 분석)입니다. 데이터의 퍼진 모양을 최대한 지키면서 많은 특징을 적은 축으로 줄입니다. 줄인 뒤에는 k-평균으로 다시 묶어 봅니다.

3. 모델 학습을 위해 해야 할 작업은 무엇인가요? → 역마다 시간대별 승하차를 "하루 승하차 가운데 %"로 바꿉니다. 40칸이 모두 같은 단위(%)이므로 표준화는 하지 않습니다. 이 표를 PCA에 넣어 주성분을 찾고, 앞의 두 주성분 점수로 지도를 그립니다.

**② 데이터 불러오기: 미니 프로젝트의 지하철 파일 불러오기**

미니 프로젝트 2단계와 같은 방법으로 한 역을 40칸 한 줄로 만듭니다. 4월은 13~14시 기록이 잘못되어 뺍니다.

In [ ]:
# ❷ 데이터 불러오기: 미니 프로젝트와 같은 지하철 파일
import os, urllib.parse, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

PATH = "서울지하철_역별_시간대별_월평균_승하차_2024.csv"
if not os.path.exists(PATH):                       # 없으면 책 깃허브 저장소에서 받기
    urllib.request.urlretrieve("https://raw.githubusercontent.com/yeonu69/bigdata-python-book/"
                               "main/ch07/data/" + urllib.parse.quote(PATH), PATH)
sw = pd.read_csv(PATH)
H = sw.loc[:, "06시이전":"24시이후"].columns         # 시간대 20칸
ok = sw[(sw["날종류"] == "평일") & (sw["월"] != 4)]   # 평일, 4월은 뺀다(미니 프로젝트 1단계)
w = ok.assign(**{h: ok[h] * ok["날수"] for h in H})
prof = (w.groupby(["역명", "구분"])[H].sum()
        .div(w.groupby(["역명", "구분"])["날수"].sum(), axis=0)
        .unstack("구분"))                             # 한 역 = 한 줄
prof.columns = [f"{b}_{h}" for h, b in prof.columns]
prof = prof[[f"{b}_{h}" for b in ["승차", "하차"] for h in H]]
prof = prof[prof.sum(axis=1) > 0]                    # 승하차가 0인 역은 뺀다
share = prof.div(prof.sum(axis=1), axis=0) * 100     # 하루 승하차 가운데 몇 %
Xp = share.to_numpy()
print("역", Xp.shape[0], "곳 × 특성", Xp.shape[1], "칸")
print("산점도로 모두 보려면:", Xp.shape[1] * (Xp.shape[1] - 1) // 2, "장")

240개 역 × 40칸입니다. 특징 둘씩 짝지어 산점도를 그리면 780장이 필요합니다.

**③ 모델 학습하기: PCA로 주성분 찾기**

`PCA( )`에 표를 넣으면 평균 빼기부터 주성분 찾기까지 한 번에 해 줍니다. `explained_variance_ratio_`에는 주성분마다 전체 분산 가운데 몇 %를 담았는지가 들어 있습니다.

In [ ]:
# ❸ 모델 학습하기: PCA로 주성분 찾기
pca = PCA().fit(Xp)                                   # 평균 빼기는 PCA가 알아서 한다
ratio = pca.explained_variance_ratio_ * 100
print("주성분마다 담은 몫(%):", ratio[:5].round(1).tolist())
print("앞에서부터 누적(%):", ratio.cumsum()[:5].round(1).tolist())

plt.figure(figsize=(6, 3))
plt.bar(range(1, 11), ratio[:10], color=["#C8234B"] * 2 + ["#9a9a9a"] * 8)
plt.plot(range(1, 11), ratio.cumsum()[:10], "o-", color="#222222", label="누적")
plt.xticks(range(1, 11))
plt.xlabel("주성분 번호")
plt.ylabel("담은 몫(%)")
plt.legend(loc="center right")
plt.show()

1주성분이 80.1%, 2주성분이 8.6%를 담습니다. 두 축만 남겨도 88.6%가 남고, 세 축이면 92.2%입니다. 몇 개를 남길지는 누적 몫이 80~90%를 넘는 곳, 또는 막대가 갑자기 낮아지는 곳(엘보우)에서 고르는 것이 보통입니다. 여기서는 그림을 그리려고 두 축을 남깁니다.

**④ 결과 확인하기: 240개 역을 2축 지도로 그리기**

`components_[0]`에는 1주성분이 40칸에 준 가중치가 들어 있습니다. `transform( )`은 역마다 새 축 위의 좌표(주성분 점수)를 돌려줍니다.

In [ ]:
# ❹ 결과 확인하기: 1주성분은 어떤 칸을 보나, 240개 역을 2축 지도로
pc1 = pd.Series(pca.components_[0], index=share.columns)
print("1주성분이 크게 본 칸:", pc1.sort_values().index[[0, 1, -2, -1]].tolist())
Zp = pca.transform(Xp)[:, :2]                          # 주성분 점수(새 좌표) 두 개
zp = pd.DataFrame(Zp, index=share.index, columns=["PC1", "PC2"]).round(1)
print(zp.loc[["신림", "잠실(송파구청)", "강남", "을지로입구", "홍대입구"]].to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"wspace": 0.35})
ax[0].plot(pc1.values[:20], color="#9a9a9a", label="승차")
ax[0].plot(pc1.values[20:], color="#C8234B", label="하차")
ax[0].axhline(0, color="#222222", linewidth=0.8)
ax[0].set_xticks([2, 8, 13, 18], ["07시", "13시", "18시", "23시"])
ax[0].set_title("1주성분의 가중치")
ax[0].legend()
ax[1].scatter(zp["PC1"], zp["PC2"], s=10, color="#9a9a9a")
for st in ["신림", "잠실(송파구청)", "강남", "을지로입구", "홍대입구", "가산디지털단지"]:
    ax[1].scatter(*zp.loc[st], s=18, color="#C8234B")
    ax[1].annotate(st.split("(")[0], zp.loc[st], xytext=(4, 3), textcoords="offset points", fontsize=10)
ax[1].set_xlabel("1주성분 (주거 ← → 일터)")
ax[1].set_ylabel("2주성분")
ax[1].set_title("240개 역의 2축 지도")
plt.show()

1주성분은 아침 7~9시 승차에 가장 큰 음수를, 아침 8~9시 하차와 저녁 6~7시 승차에 가장 큰 양수를 줍니다. 왼쪽 끝은 아침에 타고 나가는 동네 역(신림 −8.7), 오른쪽 끝은 아침에 사람이 내리는 일터 역(을지로입구 14.9)입니다. 2주성분에서는 홍대입구(−6.5)처럼 낮과 밤에도 붐비는 역이 아래로 내려갑니다.

> **[잠깐만요] 부호가 책과 반대로 나온다면**
>
> 내 컴퓨터에서 주성분 점수의 부호가 책과 반대로 나와도 괜찮습니다. 축의 방향만 뒤집혔을 뿐, 역들의 위치 관계는 같습니다.

**⑤ 결과 비교하기: 40칸으로 묶은 군집과 2축으로 묶은 군집 견주기**

40칸 그대로 k-평균(k = 3)으로 묶은 결과와, 두 주성분 점수로 묶은 결과를 표로 견줍니다. 군집 번호는 아무렇게나 붙으므로, 한 줄에 숫자가 하나씩만 있으면 두 결과가 같은 것입니다.

In [ ]:
# ❺ 결과 비교하기: 40칸으로 묶은 군집 vs 2축으로 묶은 군집
lab40 = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Xp)
lab2 = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Zp)
print(pd.crosstab(lab40, lab2, rownames=["40칸"], colnames=["2축"]).to_string())

plt.figure(figsize=(5, 3.4))
for c, col in zip(range(3), ["#9a9a9a", "#C8234B", "#222222"]):
    m = lab40 == c
    plt.scatter(Zp[m, 0], Zp[m, 1], s=10, color=col, label=f"유형 {c} ({m.sum()}곳)")
plt.xlabel("1주성분")
plt.ylabel("2주성분")
plt.legend(loc="lower left")
plt.show()

40칸으로 묶은 유형 0(107곳)은 두 축으로 묶은 유형 1과, 유형 1(48곳)은 유형 2와, 유형 2(85곳)는 유형 0과 하나도 어긋나지 않고 맞습니다. 번호만 다를 뿐 240개 역이 모두 같은 무리에 들어갔습니다. 40칸 가운데 38칸을 버렸는데도 같은 군집이 나온 것이지요. 역의 하루 리듬을 가르는 정보는 거의 다 앞의 두 축에 들어 있었다는 뜻입니다.

**배운 내용 정리하기**

**[표 7-8] PCA 문제 해결 과정**

| 단계 | 한 일 |
|---|---|
| 문제 정의하기 | 40칸짜리 하루 리듬을 2차원 평면에 그릴 수 있을까요? |
| 데이터 불러오기 | 지하철 파일을 불러와 역마다 40칸(하루 가운데 %)으로 만들기 |
| 모델 학습하기 | PCA로 주성분을 찾고, 주성분마다 담은 몫 확인하기 |
| 결과 확인하기 | 1주성분의 가중치 읽기, 240개 역을 2축 지도로 그리기 |
| 결과 비교하기 | 40칸으로 묶은 군집과 2축으로 묶은 군집 견주기 |

우리가 알게 된 정보를 정리해 봅시다.

1. 필요한 데이터는 무엇이었고, 어디에서 구했나요? → 서울교통공사의 역별 시간대별 승하차 자료를 달마다 평균 낸 파일로, 강의 자료와 책 깃허브 저장소에서 받았습니다.

2. 모델 학습에 쓴 알고리즘은 무엇이었나요? → PCA입니다. 데이터가 가장 넓게 퍼진 방향부터 새 축(주성분)을 세워 많은 특징을 적은 축으로 줄이는 방법입니다. 줄인 뒤에는 k-평균으로 다시 묶었습니다.

3. 모델 학습을 위해 어떤 작업을 했나요? → 역마다 승하차를 하루 가운데 %로 바꾸고 PCA에 넣었습니다. 두 주성분이 분산의 88.6%를 담았고, 두 축으로 묶은 군집이 40칸으로 묶은 군집과 똑같았습니다.

4. 활동을 마치며 새로 알게 된 용어를 내 말로 정리해 보세요.

**[표 7-9] 새로 알게 된 용어 정리하기**

| 용어 | 내 말로 정리하기 |
|---|---|
| 차원 축소 | |
| 주성분 분석(PCA) | |
| 주성분 | |
| 주성분 점수 | |

더 해 보기: 세 번째 주성분까지 써서 k-평균을 다시 돌리고 결과가 달라지는지 확인해 보세요. 또 `StandardScaler`로 40칸을 표준화한 뒤 PCA를 하면 1주성분의 몫이 얼마로 바뀌는지 견주어 보세요.

> **핵심 3줄**
>
> k-평균은 가장 가까운 중심에 배정하고 중심을 평균으로 옮기기를 되풀이하며, 거리를 쓰므로 열마다 단위를 맞추는 표준화가 먼저입니다.
>
> 실루엣이 높다고 군집이 진짜라는 뜻은 아니므로, 고르게 뿌린 점이나 열마다 섞은 데이터 같은 기준선과 견주어야 합니다.
>
> 지진을 k = 3으로 나누자 실루엣이 0.585였고, 깊이와 규모를 따로 섞은 데이터도 0.587로 거의 같았습니다.

## 06절의 숫자 확인하기 (선택)

책 06절의 숫자(무작위 시작 10번, 가장 먼 0.5%, 2011년 일본 지진 묶음)를 만든 코드입니다. 실습 7-1~7-10을 먼저 실행한 뒤 돌리십시오.

**S6-1**

In [ ]:
# 6.1 시작점에 따라 달라지는 답 (k = 5, 한 번씩만 돌리기)
runs = [KMeans(n_clusters=5, n_init=1, init="random", random_state=s).fit(Xs).inertia_
        for s in range(10)]
print("무작위 시작 10번의 WCSS:", sorted(np.round(runs).astype(int).tolist()))
print("서로 다른 값:", len(set(np.round(runs))), "가지")
pp = KMeans(n_clusters=5, n_init=1, init="k-means++", random_state=0).fit(Xs).inertia_
print("k-평균++ 한 번:", round(pp))

**S6-2**

In [ ]:
# 6.3 이상 탐지: 가장 가까운 중심까지의 거리가 가장 먼 0.5%
far = np.min(((Xs[:, None, :] - km.cluster_centers_[None]) ** 2).sum(axis=2), axis=1) ** 0.5
eq["중심거리"] = far
top = eq[far >= np.quantile(far, 0.995)].sort_values("중심거리", ascending=False)
print("가장 먼 0.5%:", len(top), "건")
print(top[["time", "mag", "depth", "region"]].head(6).to_string(index=False))

**S6-3**

In [ ]:
# 6.2 여진 묶기: 2011년 일본 주변 규모 5 이상 지진을 위치(100km)와 시간(30일)으로
win = eq[(year == 2011) & eq["latitude"].between(25, 50) & eq["longitude"].between(125, 155)].copy()
main = win.loc[win["mag"].idxmax()]
xk = (win["longitude"] - main["longitude"]) * 111.2 * np.cos(np.radians(38)) / 100
yk = (win["latitude"] - main["latitude"]) * 111.2 / 100
tk = (win["time"] - pd.Timestamp("2011-01-01", tz="UTC")).dt.total_seconds() / 86400 / 30
win["st"] = DBSCAN(eps=0.6, min_samples=10).fit(np.column_stack([xk, yk, tk])).labels_
seq = win[win["st"] == win.loc[main.name, "st"]]
after = seq["time"] - main["time"]
print("2011년 일본 주변:", len(win), "건, 본진", main["time"].date(), "규모", main["mag"])
print("본진과 한 묶음:", len(seq), "건, 첫날", ((after >= pd.Timedelta(0)) & (after < pd.Timedelta(days=1))).sum(),
      "건, 첫 주", ((after >= pd.Timedelta(0)) & (after < pd.Timedelta(days=7))).sum(),
      "건, 한 달 뒤부터", (after >= pd.Timedelta(days=30)).sum(), "건, 전진", (after < pd.Timedelta(0)).sum(), "건")

### 🗺️ [지도로 보기 3] 2011년 동일본 대지진의 묶음

S6-3에서 위치와 시간으로 묶은 2011년 일본 주변 지진을 지도에 확대해 봅니다. 확대하면 해안선이 거칠어지므로, 이번에는 더 자세한 1:5천만 축척 지도(약 1.6MB)를 씁니다. 큰 별이 본진입니다.

In [ ]:
land50 = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
                       "v5.1.2/geojson/ne_50m_land.geojson")
fig, ax = plt.subplots(figsize=(6, 6))
land50.plot(ax=ax, color="#ececec", edgecolor="#9a9a9a", linewidth=0.6, zorder=0)
other = win[win["st"] != win.loc[main.name, "st"]]
ax.scatter(other["longitude"], other["latitude"], s=8, color="#8a8a8a", label=f"다른 지진 {len(other)}건")
ax.scatter(seq["longitude"], seq["latitude"], s=8, color="#E04B3C", alpha=0.6, label=f"본진 묶음 {len(seq)}건")
ax.scatter(main["longitude"], main["latitude"], marker="*", s=400, color="gold", edgecolor="k", label="본진")
ax.set_xlim(125, 155)
ax.set_ylim(25, 50)
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.set_title("2011년 일본 주변 규모 5 이상 지진")
ax.legend(loc="lower right")
plt.show()

dist_km = np.hypot(xk[seq.index], yk[seq.index]) * 100    # 본진에서 떨어진 거리(km)
print("본진 묶음의 거리(km): 가운데", round(dist_km.median()), " 90%가", round(dist_km.quantile(0.9)), "km 안")
print("본진 묶음 가운데 바다 위 진앙:", round((~seq["육지"]).mean() * 100, 1), "%")

🔍 **지도 읽기**: 본진 묶음 596건은 본진을 중심으로 혼슈 동쪽 바다에 빽빽하게 모여 있습니다. 본진에서 떨어진 거리의 가운데 값이 160km이고 90%가 272km 안이며, 96.3%가 바다 위입니다. 같은 해 같은 지역에서 난 지진이라도 류큐 열도나 홋카이도 쪽의 회색 점들은 다른 묶음입니다. 위치와 시간을 함께 쓰는 DBSCAN이 "이 본진의 여진"만 골라낸 것을 지도로 확인할 수 있습니다.

## [미니 프로젝트 7] 지하철역은 몇 가지 하루를 살까

**질문.** 아침 8시, 어떤 역에서는 사람들이 쏟아져 타고, 어떤 역에서는 쏟아져 내립니다. 서울 지하철역을 시간대별 승하차의 "하루 리듬"만으로 묶으면 몇 가지 유형이 나올까요? 그 유형은 진짜일까요?

**데이터.** 서울교통공사가 공공데이터포털에 올린 "서울교통공사_역별 일별 시간대별 승하차인원 정보" 2024년치입니다. 원자료는 하루, 한 역, 한 호선의 승차(또는 하차)가 한 행이고 199,424행입니다. 이 책에서는 이것을 달마다 평균 낸 파일(서울지하철_역별_시간대별_월평균_승하차_2024.csv)을 함께 나누어 줍니다. 11,506행 × 26열이고, 한 행은 "어느 달, 평일 또는 쉬는 날, 한 역의 승차(또는 하차)를 시간대별로 평균한 값"입니다. 환승역은 호선을 합쳐 역 이름 하나로 셌습니다. 역 이름은 원자료 그대로여서, 4호선은 총신대입구(이수), 7호선은 이수로 적힌 역은 두 역으로 셉니다. 파일을 Colab 왼쪽 파일 창에 끌어다 놓고 시작합니다.


**1단계: 데이터 신원 확인**([실습 예제 7-1] 응용). 파일을 읽어 빈칸과 역마다의 줄 수를 세고, 달마다 시간대 합계를 견주시오.

In [ ]:
# 파일이 없으면 업로드 창을 엽니다 (코랩). 책과 함께 받은 CSV 파일을 고르십시오.
import os
if not os.path.exists("서울지하철_역별_시간대별_월평균_승하차_2024.csv"):
    from google.colab import files
    files.upload()

In [ ]:
# 데이터 바꾸기: 다른 해의 파일을 쓸 때는 이 두 줄만 고칩니다.
PATH = "서울지하철_역별_시간대별_월평균_승하차_2024.csv"   # Colab 왼쪽 파일 창에 올린 파일
HOURS = slice("06시이전", "24시이후")                    # 시간대 열의 처음과 끝

sw = pd.read_csv(PATH)
print(sw.shape, "/ 빈칸", sw.isna().sum().sum(), "/ 역", sw["역명"].nunique(), "곳")
per_st = sw.groupby("역명").size()                    # 역마다 몇 줄인가 (12달 × 2 × 2 = 48)
print(per_st.value_counts().to_dict(), per_st[per_st != 48].to_dict())
H = sw.loc[:, HOURS].columns
month = sw[sw["날종류"] == "평일"].groupby("월")[H].sum() / 1000   # 달별 시간대 합(천 명)
print(month[["12-13시", "13-14시", "14-15시"]].round(0).astype(int).T.to_string())

month.T.plot(legend=False, color="#9a9a9a", figsize=(7, 3))
plt.plot(month.loc[4].values, color="#E04B3C", linewidth=2)   # 4월만 빨갛게
plt.xticks(range(len(H)), H, rotation=90)
plt.ylabel("평일 하루 평균(천 명)")
plt.show()

역 241곳 가운데 239곳은 48줄(12달 × 평일·쉬는 날 × 승차·하차)인데 두 곳은 아닙니다. 암사역사공원(22줄)은 2024년 하반기에 새로 연 역이고, 까치울(12줄)은 기록이 일부 달에만 있습니다. 더 큰 문제는 4월입니다. 1장과 3장에서 만난 2024년 4월 13~14시 기록입니다. 13~14시 합계가 4,243천 명으로 다른 달(391~450천 명)의 10배쯤입니다. 원자료를 열어 보면 4월의 13~14시 칸에만 앞뒤 시간대의 몇 배에서 열 배가 넘는 값이 들어간 날이 많습니다. 사람들이 갑자기 그 시간에만 몰렸을 리는 없으니 기록의 잘못으로 보고, 4월은 빼고 씁니다. 13장에서도 이 4월 기록을 "멈추고 기록하기"의 예로 다시 만납니다.

**2단계: 기준선 만들기**([실습 예제 7-4], [7-6] 응용). 평일 하루 리듬을 역마다 한 줄로 만드시오. 인원수를 그대로 묶은 결과와, 열마다 따로 섞은 데이터의 실루엣을 기준선으로 구하시오.

In [ ]:
ok = sw[(sw["날종류"] == "평일") & (sw["월"] != 4)]          # 4월은 뺀다
w = ok.assign(**{h: ok[h] * ok["날수"] for h in H})           # 날수만큼 가중
prof = (w.groupby(["역명", "구분"])[H].sum()
        .div(w.groupby(["역명", "구분"])["날수"].sum(), axis=0)
        .unstack("구분"))                                     # 한 역 = 한 줄
prof.columns = [f"{b}_{h}" for h, b in prof.columns]
prof = prof[[f"{b}_{h}" for b in ["승차", "하차"] for h in H]]
total = prof.sum(axis=1)
print("승하차가 0인 역:", total[total == 0].index.tolist())
prof, total = prof[total > 0], total[total > 0]
share = prof.div(total, axis=0) * 100                         # 하루 승하차 가운데 몇 %인가
print(len(share), "개 역 ×", share.shape[1], "칸,  가장 붐비는 역:",
      total.idxmax(), round(total.max()))

# 기준선 1: 인원수를 그대로 묶으면?
k_raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit(prof)
print("인원수 그대로 묶은 군집의 평균 하루 승하차:",
      total.groupby(k_raw.labels_).agg(["size", "mean"]).round(0).astype(int).values.tolist())
# 기준선 2: 무리가 없는 데이터 (시간대 열마다 따로 섞기)
Xm = share.to_numpy()
rng2 = np.random.default_rng(0)
Sm = np.column_stack([rng2.permutation(Xm[:, j]) for j in range(Xm.shape[1])])
for k in range(2, 7):
    m1 = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xm)
    m2 = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Sm)
    print(f"k={k}: 실제 실루엣 {silhouette_score(Xm, m1.labels_):.3f}, "
          f"섞은 데이터 {silhouette_score(Sm, m2.labels_):.3f}")

승하차가 0으로만 적힌 역 하나를 빼면 240개 역이 남습니다. 인원수를 그대로 묶으면 하루 평균 2만 5천 명, 6만 7천 명, 11만 8천 명인 역끼리 모입니다. 리듬이 아니라 크기로 나뉜 것입니다. 3.3에서 깊이가 거리를 독차지한 것과 같은 일입니다. 그래서 역마다 "하루 승하차 가운데 그 시간대가 몇 %인가"로 바꾸었습니다. 이렇게 바꾼 데이터의 실루엣은 k = 2에서 0.493, k = 3에서 0.388로, 열마다 섞은 데이터(0.167, 0.120)보다 세 배쯤 높습니다.

**3단계: 유형 찾고 이름 붙이기**([실습 예제 7-5] 응용). k = 3으로 묶고, 중심의 하루 리듬을 그려 유형마다 이름을 붙이시오.

In [ ]:
K = 3
kmm = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xm)
cent = pd.DataFrame(kmm.cluster_centers_, columns=share.columns)
am = cent[["승차_07-08시", "승차_08-09시"]].sum(axis=1) - cent[["하차_07-08시", "하차_08-09시"]].sum(axis=1)
print("아침 7~9시 (승차 % − 하차 %):", am.round(1).tolist())
share["유형"] = kmm.labels_
for c in range(K):
    names = share.index[share["유형"] == c]
    top = total[names].sort_values(ascending=False).head(5).index.tolist()
    print(f"유형 {c}: {len(names)}개 역, 붐비는 역 {top}")

fig, ax = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
for c, col in zip(range(K), ["#9a9a9a", "#E04B3C", "#222222"]):
    ax[0].plot(cent.iloc[c, :len(H)].values, color=col, label=f"유형 {c}")
    ax[1].plot(cent.iloc[c, len(H):].values, color=col)
for a_, t in zip(ax, ["승차", "하차"]):
    a_.set_title(t)
    a_.set_xticks([2, 8, 13, 18], ["07시", "13시", "18시", "23시"])
ax[0].set_ylabel("하루 가운데 %")
ax[0].legend(loc="upper center")
plt.show()

실루엣은 k = 2가 가장 높았지만, k = 3을 골랐습니다. 지진에서처럼 쓰임새를 생각했습니다. 유형 0(107개 역)은 아침 7~9시에 타는 사람이 내리는 사람보다 하루의 11.7%p 많습니다. 신림, 수유, 쌍문처럼 사람이 사는 동네의 역입니다. 유형 1(48개 역)은 거꾸로 13.2%p 적습니다. 강남, 서울역, 역삼처럼 일터가 모인 역입니다. 유형 2(85개 역)는 아침의 타고 내림이 거의 같습니다. 잠실, 홍대입구, 고속터미널처럼 사는 사람과 찾아오는 사람이 섞인 역입니다.

**4단계: 군집에 쓰지 않은 정보로 확인하기.** 쉬는 날 하루 승하차를 평일로 나눈 값을 유형마다 견주시오. 일터가 모인 역이라면 쉬는 날에는 한산해야 합니다.

In [ ]:
# 군집에 쓰지 않은 정보로 확인하기: 쉬는 날 승하차 ÷ 평일 승하차
offday = sw[(sw["날종류"] == "쉬는날") & (sw["월"] != 4)]
days = offday.groupby("역명")["날수"].sum() / 2               # 승차·하차 두 줄에 날수가 같이 적혀 있다
off = offday.assign(t=offday[H].sum(axis=1) * offday["날수"]).groupby("역명")["t"].sum() / days
ratio = (off[share.index] / total).rename("쉬는날÷평일")
print(ratio.groupby(share["유형"]).describe()[["count", "25%", "50%", "75%"]].round(2).to_string())

쉬는 날 승하차는 평일의 몇 배일까요? 가운데 값이 유형 1은 0.48, 유형 0은 0.60, 유형 2는 0.64입니다. 쉬는 날 숫자는 군집을 만들 때 쓰지 않았습니다. 그런데도 "일터가 모인 역"은 쉬는 날 평일의 절반 아래로 줄었습니다. 스노가 물 공급 회사로 확인했듯이, 이름이 들어맞는다는 증거 하나를 얻은 것입니다.

**5단계: 한계 쓰기.** 이 유형이 아직 모르는 것을 [표 7-5]처럼 표로 쓰시오. 이 파일에는 서울교통공사가 운영하는 1~8호선 구간만 들어 있습니다. 9호선이나 신분당선만 지나는 역은 없다는 점을 꼭 다루십시오.

**제출물.** 그림 2장(달별 시간대 합계, 유형별 하루 리듬), 숫자 표 1개(k별 실루엣과 섞은 데이터의 실루엣, 유형별 쉬는 날 비율), 결론 5문장 이내.

**[표 7-13] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 4월의 13~14시 값, 줄 수가 다른 역, 0인 역을 찾아 처리 방법을 밝힘 | 빈칸만 확인함 | 점검 없이 바로 묶음 |
| 기준선 비교(20%) | 인원수 그대로 묶은 결과와 섞은 데이터의 실루엣을 모두 견줌 | 하나만 견줌 | 기준선 없이 군집만 씀 |
| 기법 적용(30%) | 비율로 바꾼 까닭을 설명하고 k를 지표와 쓰임새로 고름 | 군집은 맞으나 k를 고른 까닭이 없음 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 쉬는 날 비율로 이름을 확인하고 한계를 표로 셋 이상 씀 | 이름만 붙이고 한계를 한두 가지 씀 | 숫자만 나열함 |

**교수용 정답 예시.** 핵심 숫자는 이렇습니다. 11,506행, 역 241곳(까치울 12줄, 암사역사공원 22줄), 4월 13~14시 합계 4,243천 명(다른 달 391~450천 명)이라 4월을 뺍니다. 승하차가 0인 까치울을 빼면 240개 역입니다. 인원수 그대로 묶으면 하루 2만 5천, 6만 7천, 11만 8천 명처럼 크기로 나뉩니다. 비율로 바꾼 뒤 실루엣은 k = 2가 0.493, k = 3이 0.388이고, 섞은 데이터는 0.167, 0.120입니다. k = 3의 유형은 아침에 타는 동네 역 107곳(아침 승차 − 하차 +11.7%p), 아침에 내리는 일터 역 48곳(−13.2%p), 둘 사이 85곳(+0.3%p)입니다. 쉬는 날 ÷ 평일의 가운데 값은 0.60, 0.48, 0.64로, 군집에 쓰지 않은 정보가 이름을 받쳐 줍니다. 결론의 예: "서울 지하철역의 평일 리듬은 사는 곳, 일하는 곳, 둘이 섞인 곳으로 나뉜다. 일터 역은 쉬는 날 평일의 절반 아래로 줄어 이름이 맞는다. 다만 9호선과 다른 회사 구간이 빠졌고, 2024년 한 해만 보았다."

> **[더 알아보기] 공공데이터포털에서 원자료를 받아 직접 만들기 ☆**
>
> 공공데이터포털(data.go.kr)에서 "서울교통공사 역별 일별 시간대별 승하차인원"을 찾으면 해마다 CSV 파일을 받을 수 있습니다. 받은 파일은 한글이 cp949로 저장되어 있어서 `encoding="cp949"`를 붙여 읽어야 합니다. 해마다 열 이름과 개수가 조금씩 다르니 먼저 `columns`를 찍어 보십시오. 2024년 파일에서 이 책의 파일을 만든 코드는 상자 아래에 실었습니다. 쉬는 날은 토요일, 일요일과 `holidays.KR( )`이 알려 주는 공휴일입니다. 같은 역이 호선마다 따로 적혀 있으므로 역 이름으로 더한 뒤 평균을 냅니다.
>
> 다른 해의 파일로 만들면 1단계의 `PATH` 한 줄만 바꾸어 같은 단계를 되풀이할 수 있습니다.

**[더 알아보기] 코드** (원자료 CSV를 올린 뒤 새 코드 셀에 붙여 넣어 실행하십시오. 실행하면 책의 파일과 똑같은 파일을 새로 만듭니다.)

```python
# [더 알아보기] 공공데이터포털에서 받은 원자료로 이 책의 CSV 파일 만들기
import holidays                                    # 공휴일 목록 (없으면 !pip install holidays)
RAW = "서울교통공사_역별 일별 시간대별 승하차인원 정보_20241231.csv"
raw = pd.read_csv(RAW, encoding="cp949")           # 한글이 cp949로 저장된 파일
print(raw.shape)
hours = raw.columns[6:].tolist()                   # '06시이전' ~ '24시이후'
raw["날짜"] = pd.to_datetime(raw["수송일자"])
kr = holidays.KR(years=2024)
off_day = (raw["날짜"].dt.dayofweek >= 5) | raw["날짜"].dt.date.map(lambda x: x in kr)
raw["날종류"] = np.where(off_day, "쉬는날", "평일")
raw["월"] = raw["날짜"].dt.month
# 환승역은 호선마다 따로 적혀 있으므로 역명으로 더한 뒤, 달·날종류마다 하루 평균
daily = raw.groupby(["역명", "날짜", "월", "날종류", "승하차구분"])[hours].sum().reset_index()
m = daily.groupby(["월", "날종류", "역명", "승하차구분"])[hours].mean().round(1).reset_index()
ndays = daily.groupby(["월", "날종류"])["날짜"].nunique().rename("날수").reset_index()
lines = (raw.groupby("역명")["호선"].unique()
         .map(lambda a: ",".join(sorted(a, key=lambda t: int(t[0])))).rename("호선"))
m = m.merge(ndays, on=["월", "날종류"]).merge(lines, left_on="역명", right_index=True)
m = m.rename(columns={"승하차구분": "구분"})
m.columns = [c.replace("시간대", "시") for c in m.columns]
m = m[["월", "날종류", "날수", "역명", "호선", "구분"] + [h.replace("시간대", "시") for h in hours]]
m.to_csv("서울지하철_역별_시간대별_월평균_승하차_2024.csv", index=False)
print(m.shape)
```

## ✏️ [연습문제 7-6] DBSCAN의 설정 바꾸기

[실습 예제 7-10]에서 `eps`를 200km, `min_samples`를 15로 바꿔 다시 돌리고, 밀집 지대의 수, 가장 큰 지대의 지진 수, 노이즈 비율을 구하시오. 결과가 어떻게 달라졌는지 설명하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
db2 = DBSCAN(eps=200 / 6371, min_samples=15,
             metric="haversine", algorithm="ball_tree").fit(rad)
lab = db2.labels_
sizes = pd.Series(lab[lab >= 0]).value_counts()
print("밀집 지대:", len(sizes), "곳, 가장 큰 지대:", sizes.iloc[0], "건,",
      "노이즈:", round((lab == -1).mean() * 100, 1), "%")
print("가장 큰 지대의 지역:", big["region"][lab == sizes.index[0]].value_counts().head(4).to_dict())
ax = world_map("반지름 200km, 15개로 바꾼 뒤 가장 큰 지대")
ax.scatter(big["longitude"], big["latitude"], s=2, color="#b0b0b0")
one = big[lab == sizes.index[0]]
ax.scatter(one["longitude"], one["latitude"], s=4, color="#E04B3C")
plt.show()